# BLaIR-style Dual Encoder Product Retrieval
**Amazon Applied Scientist Portfolio Project**

End-to-end dense retrieval pipeline on Amazon Reviews 2023 (Electronics).  
Run cells 1–12 in order on a GPU GPU instance.

| System | NDCG@10 (expected) |
|--------|--------------------|
| BM25 Okapi | ~0.02–0.60 |
| Zero-shot BiEncoder | ~0.15–0.25 |
| BiEncoder (fine-tuned) | ~0.55–0.70 |
| DualEncoder (random neg) | ~0.60–0.73 |
| DualEncoder (BM25 hard-neg) ★ | ~0.70–0.82 |
| Hybrid BM25 + Dense (RRF) | ~0.73–0.85 |

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Setup
# ═══════════════════════════════════════════════════════════
import subprocess, sys

# Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'rank-bm25', 'faiss-gpu', 'transformers', 'datasets',
    'tqdm', 'sentence-transformers', 'pyarrow'], check=True)

import os, torch
from pathlib import Path

# Create required directories
for d in ['data', 'results/significance', 'results/error_analysis',
          'artifacts/models', 'artifacts/cache']:
    Path(d).mkdir(parents=True, exist_ok=True)

# Verify GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — training will be very slow')
print('Setup complete.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Build Dataset
# Downloads Amazon Reviews 2023 Electronics, 20k sample,
# product-level 80/10/10 split.
# ═══════════════════════════════════════════════════════════
import subprocess
result = subprocess.run(
    ['python', 'build_dataset.py', '--n-samples', '100000', '--seed', '42',
     '--data-dir', 'data/'],
    capture_output=False
)
assert result.returncode == 0, 'build_dataset.py failed'

import pandas as pd
from pathlib import Path
for name in ['train', 'val', 'test', 'corpus']:
    df = pd.read_parquet(f'data/{name}.parquet')
    print(f'{name:8}: {len(df):6,} rows | cols: {list(df.columns)}')

# Verify no product leakage
train_df = pd.read_parquet('data/train.parquet')
test_df  = pd.read_parquet('data/test.parquet')
overlap  = set(train_df['product_id']) & set(test_df['product_id'])
print(f'\nProduct leakage (must be 0): {len(overlap)}')
assert len(overlap) == 0, 'FAIL: product leakage detected'

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — BM25 Baseline
# Keyword retrieval using BM25 Okapi.
# Establishes lexical baseline before any dense models.
# ═══════════════════════════════════════════════════════════
import subprocess
result = subprocess.run(
    ['python', 'evaluate_bm25.py',
     '--data-dir', 'data/',
     '--output-dir', 'results/bm25/',
     '--cache', '--cache-path', 'artifacts/cache/bm25.pkl'],
    capture_output=False
)
assert result.returncode == 0, 'evaluate_bm25.py failed'

import json
with open('results/bm25/metrics.json') as f:
    bm25_metrics = json.load(f)
print('\nBM25 Results:')
for k, v in sorted(bm25_metrics.items()):
    if isinstance(v, float):
        print(f'  {k:<20}: {v:.4f}')
    else:
        print(f'  {k:<20}: {v}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Zero-Shot Dense Retrieval
# bert-base-uncased with NO fine-tuning.
# Shows how much pre-training helps before task-specific training.
# ═══════════════════════════════════════════════════════════
import subprocess
result = subprocess.run(
    ['python', 'evaluate_dense.py',
     '--data-dir', 'data/',
     '--output-dir', 'results/zeroshot/',
     '--model-type', 'biencoder'],
    capture_output=False
)
assert result.returncode == 0, 'evaluate_dense.py (zero-shot) failed'

import json
with open('results/zeroshot/metrics.json') as f:
    zs = json.load(f)
print(f"\nZero-shot NDCG@10 : {zs['ndcg@10']:.4f}")
print(f"Zero-shot R@10    : {zs['recall@10']:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — Train Bi-Encoder (shared weights) + Evaluate
# Single BERT for both query and product.
# Random in-batch negatives, seed=42.
# ═══════════════════════════════════════════════════════════
import subprocess

# Train
r = subprocess.run(
    ['python', 'train.py',
     '--model-type', 'biencoder', '--neg-mode', 'random',
     '--seed', '42', '--epochs', '5', '--batch-size', '16',
     '--output-dir', 'artifacts/models/biencoder_seed42/'],
    capture_output=False
)
assert r.returncode == 0, 'train.py (biencoder) failed'

# Evaluate
r = subprocess.run(
    ['python', 'evaluate_dense.py',
     '--data-dir', 'data/',
     '--checkpoint', 'artifacts/models/biencoder_seed42/best_model',
     '--output-dir', 'results/biencoder/'],
    capture_output=False
)
assert r.returncode == 0, 'evaluate_dense.py (biencoder) failed'

import json
with open('results/biencoder/metrics.json') as f:
    bi = json.load(f)
print(f"\nBiEncoder NDCG@10 : {bi['ndcg@10']:.4f}")
print(f"BiEncoder R@10    : {bi['recall@10']:.4f}")
print(f"Latency (ms/q)    : {bi.get('latency_ms', 'N/A')}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Train Dual Encoder (separate weights) + Evaluate
# BLaIR-style: separate BERT for query and product towers.
# Random negatives (ablation vs hard-neg in Cell 7).
# ═══════════════════════════════════════════════════════════
import subprocess

# Train
r = subprocess.run(
    ['python', 'train.py',
     '--model-type', 'dual', '--neg-mode', 'random',
     '--seed', '42', '--epochs', '5', '--batch-size', '16',
     '--output-dir', 'artifacts/models/dual_seed42/'],
    capture_output=False
)
assert r.returncode == 0, 'train.py (dual, random) failed'

# Evaluate
r = subprocess.run(
    ['python', 'evaluate_dense.py',
     '--data-dir', 'data/',
     '--checkpoint', 'artifacts/models/dual_seed42/best_model',
     '--output-dir', 'results/dual/'],
    capture_output=False
)
assert r.returncode == 0, 'evaluate_dense.py (dual) failed'

import json
with open('results/dual/metrics.json') as f:
    dual = json.load(f)
print(f"\nDualEncoder NDCG@10 : {dual['ndcg@10']:.4f}")
print(f"DualEncoder R@10    : {dual['recall@10']:.4f}")
print(f"Latency (ms/q)      : {dual.get('latency_ms', 'N/A')}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Dual Encoder + BM25 Hard Negatives (main result)
# Hard negatives: BM25 top results excluding the true positive.
# Forces fine-grained semantic discrimination.
# ═══════════════════════════════════════════════════════════
import subprocess

# Train (BM25 hard-neg mode)
r = subprocess.run(
    ['python', 'train.py',
     '--model-type', 'dual', '--neg-mode', 'bm25',
     '--seed', '42', '--epochs', '5', '--batch-size', '16',
     '--output-dir', 'artifacts/models/dual_hardneg_seed42/'],
    capture_output=False
)
assert r.returncode == 0, 'train.py (dual, bm25) failed'

# Evaluate
r = subprocess.run(
    ['python', 'evaluate_dense.py',
     '--data-dir', 'data/',
     '--checkpoint', 'artifacts/models/dual_hardneg_seed42/best_model',
     '--output-dir', 'results/dual_hardneg/'],
    capture_output=False
)
assert r.returncode == 0, 'evaluate_dense.py (dual_hardneg) failed'

import json
with open('results/dual_hardneg/metrics.json') as f:
    hn = json.load(f)
print(f"\nDual+HardNeg NDCG@10 : {hn['ndcg@10']:.4f}")
print(f"Dual+HardNeg R@10    : {hn['recall@10']:.4f}")
print(f"Latency (ms/q)       : {hn.get('latency_ms', 'N/A')}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — Multi-Seed Reproducibility (seeds 123, 456)
# Trains the best model (dual + BM25 hard-neg) with seeds
# 123 and 456, then reports mean ± std across seeds 42, 123, 456.
# ═══════════════════════════════════════════════════════════
import subprocess, json, numpy as np
from pathlib import Path

for seed in [123, 456]:
    print(f'\n--- Training seed={seed} ---')
    r = subprocess.run(
        ['python', 'train.py',
         '--model-type', 'dual', '--neg-mode', 'bm25',
         '--seed', str(seed), '--epochs', '5', '--batch-size', '16',
         '--output-dir', f'artifacts/models/dual_hardneg_seed{seed}/'],
        capture_output=False
    )
    assert r.returncode == 0, f'train.py seed={seed} failed'

    r = subprocess.run(
        ['python', 'evaluate_dense.py',
         '--data-dir', 'data/',
         '--checkpoint', f'artifacts/models/dual_hardneg_seed{seed}/best_model',
         '--output-dir', f'results/dual_hardneg_seed{seed}/'],
        capture_output=False
    )
    assert r.returncode == 0, f'evaluate_dense.py seed={seed} failed'

# Compute mean ± std across seeds 42, 123, 456
metrics_keys = ['ndcg@10', 'recall@10', 'mrr', 'recall@1']
all_results  = {}
for seed in [42, 123, 456]:
    p = Path(f'results/dual_hardneg_seed{seed}/metrics.json')
    if not p.exists():
        p = Path('results/dual_hardneg/metrics.json')  # seed 42 saved here
    if p.exists():
        with open(p) as f:
            all_results[seed] = json.load(f)

print('\n=== MULTI-SEED RESULTS (Dual + BM25 Hard-Neg) ===')
print(f'{"Seed":<8}', ' '.join(f'{k:>12}' for k in metrics_keys))
print('-' * 60)
for seed, m in all_results.items():
    row = ' '.join(f'{m.get(k, 0):>12.4f}' for k in metrics_keys)
    print(f'{seed:<8} {row}')
print('-' * 60)

seed_stats = {}
for k in metrics_keys:
    vals = [all_results[s].get(k, 0) for s in all_results]
    mean_v, std_v = np.mean(vals), np.std(vals)
    seed_stats[k] = {'mean': round(float(mean_v), 4), 'std': round(float(std_v), 4)}
    print(f'{k:<12}: {mean_v:.4f} ± {std_v:.4f}')

with open('results/seed_stats.json', 'w') as f:
    json.dump({'seeds': [42, 123, 456], 'metrics': seed_stats}, f, indent=2)
print('\nSaved: results/seed_stats.json')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — All McNemar Significance Tests
# Four pairwise comparisons as required by spec:
#   1. BiEncoder vs BM25
#   2. Dual vs BiEncoder
#   3. HardNeg vs Random
#   4. Hybrid vs DualHardNeg (run after Cell 10)
# ═══════════════════════════════════════════════════════════
import subprocess
from pathlib import Path

comparisons = [
    ('results/biencoder',   'results/bm25',         'BiEncoder', 'BM25'),
    ('results/dual',        'results/biencoder',     'Dual_Random', 'BiEncoder'),
    ('results/dual_hardneg','results/dual',           'Dual_HardNeg', 'Dual_Random'),
]

for dir_a, dir_b, label_a, label_b in comparisons:
    if not Path(dir_a + '/per_query_metrics.parquet').exists():
        print(f'SKIP {label_a} vs {label_b} — missing per_query_metrics')
        continue
    r = subprocess.run(
        ['python', 'run_mcnemar.py',
         '--a', dir_a, '--b', dir_b,
         '--label-a', label_a, '--label-b', label_b],
        capture_output=False
    )
    assert r.returncode == 0, f'McNemar {label_a} vs {label_b} failed'

# Note: Hybrid vs DualHardNeg is run at the end of Cell 10
print('\nAll available McNemar tests complete.')
print('Significance files saved to: results/significance/')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — Hybrid BM25 + Dense (RRF) + Cross-Encoder Reranker
# RRF fusion combines lexical and semantic retrieval.
# Cross-encoder reranks the top-10 candidates.
# Then runs McNemar: Hybrid vs DualHardNeg.
# ═══════════════════════════════════════════════════════════
import subprocess
from pathlib import Path

# Hybrid retrieval
r = subprocess.run(
    ['python', 'evaluate_hybrid.py',
     '--data-dir', 'data/',
     '--checkpoint', 'artifacts/models/dual_hardneg_seed42/best_model',
     '--output-dir', 'results/hybrid/'],
    capture_output=False
)
assert r.returncode == 0, 'evaluate_hybrid.py failed'

# Cross-encoder reranker
r = subprocess.run(
    ['python', 'evaluate_reranker.py',
     '--stage1-dir', 'results/hybrid/',
     '--output-dir', 'results/reranker/'],
    capture_output=False
)
if r.returncode != 0:
    print('WARNING: evaluate_reranker.py failed (non-critical, continuing)')

# McNemar: Hybrid vs DualHardNeg (4th comparison)
if Path('results/hybrid/per_query_metrics.parquet').exists():
    r = subprocess.run(
        ['python', 'run_mcnemar.py',
         '--a', 'results/hybrid/',
         '--b', 'results/dual_hardneg/',
         '--label-a', 'Hybrid_RRF',
         '--label-b', 'Dual_HardNeg'],
        capture_output=False
    )
    assert r.returncode == 0, 'McNemar Hybrid vs DualHardNeg failed'

import json
if Path('results/hybrid/metrics.json').exists():
    with open('results/hybrid/metrics.json') as f:
        hyb = json.load(f)
    print(f"\nHybrid NDCG@10   : {hyb['ndcg@10']:.4f}")
    print(f"Hybrid R@10      : {hyb['recall@10']:.4f}")
    print(f"Latency (ms/q)   : {hyb.get('latency_ms', 'N/A')}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — Error Analysis
# Categorize 50 worst failures from best model.
# Categories: lexical_mismatch, too_short_query, ambiguous_query,
#             rare_product, wrong_but_reasonable
# ═══════════════════════════════════════════════════════════
import subprocess
from pathlib import Path

r = subprocess.run(
    ['python', 'run_error_analysis.py',
     '--results-dir', 'results/dual_hardneg/',
     '--corpus', 'data/corpus.parquet',
     '--train', 'data/train.parquet',
     '--output-dir', 'results/error_analysis/'],
    capture_output=False
)
if r.returncode != 0:
    print('WARNING: run_error_analysis.py returned non-zero (check args)')

# Display results
import json
import pandas as pd

cat_path = Path('results/error_analysis/error_categories.csv')
if cat_path.exists():
    cats = pd.read_csv(cat_path)
    print('\nError Category Distribution (50 worst failures):')
    print(cats.to_string(index=False))
else:
    print('Error categories file not found — check run_error_analysis.py')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 12 — Final Results Table
# Prints full leaderboard with NDCG@10, Recall@10, MRR,
# Recall@1, Latency, and McNemar p-values for all systems.
# Saves: results/comparison_table.csv, results/table.tex
# ═══════════════════════════════════════════════════════════
import subprocess
r = subprocess.run(
    ['python', 'generate_table.py',
     '--results-dir', 'results/',
     '--sig-dir', 'results/significance/',
     '--output-dir', 'results/'],
    capture_output=False
)
assert r.returncode == 0, 'generate_table.py failed'

# Display saved table
import pandas as pd
from pathlib import Path

csv_path = Path('results/comparison_table.csv')
if csv_path.exists():
    print('\nFinal Comparison Table:')
    print(pd.read_csv(csv_path).to_string(index=False))

# Print seed stability summary
import json
seed_path = Path('results/seed_stats.json')
if seed_path.exists():
    with open(seed_path) as f:
        ss = json.load(f)
    print(f'\nBest model (Dual + BM25 hard-neg) mean ± std across seeds {ss["seeds"]}:')
    for k, v in ss['metrics'].items():
        print(f'  {k:<12}: {v["mean"]:.4f} ± {v["std"]:.4f}')

print('\n[DONE] All 12 cells complete. Results saved to results/')